In [ ]:
# finetune_flan_t5_small.py
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments

model_name = "google/flan-t5-small"
dataset_path = "train.jsonl"   # your instruction dataset

# 1. Load dataset
dataset = load_dataset("json", data_files={"train": dataset_path, "validation": dataset_path})

# 2. Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Preprocess function
def preprocess(example):
    instruction = example["instruction"]
    inp = example["input"]
    tgt = example["output"]

    source = f"Instruction: {instruction}\nInput: {inp}"
    model_inputs = tokenizer(source, max_length=256, truncation=True)
    labels = tokenizer(tgt, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(preprocess, batched=False)

# 4. Data collator & training args
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = TrainingArguments(
    output_dir="./flan_t5_small_finetuned",
    evaluation_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    logging_steps=10,
)

# 5. Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
)

trainer.train()
trainer.save_model("./flan_t5_small_finetuned")


Generating train split: 20 examples [00:00, 4794.31 examples/s]
Generating validation split: 20 examples [00:00, 8584.33 examples/s]


In [2]:
!export HF_HUB_ENABLE_HF_TRANSFER=1
